In [1]:
//%jsroot on

#include "TCutG.h"

In [2]:
#define _15C 28
#define NUM 100
#define _N 6

In [3]:
/******************/
Int_t stripnum=1;
/******************/

In [4]:
int start = 16;
int stop = 24;
TChain *fch;
fch = new TChain("tree", "tree");
for (int i= start; i<=stop; i++){
  TString fileName = TString::Format("/home/long/data/25e04/10Be/hit/run%02d_hit.root" , i);
  fch->AddFile(fileName.Data());
}
// fch->Print();

In [5]:
TCanvas *c1=new TCanvas("c1","c1");

Double_t silicon_thickness = 284.; // Microns
string ss;
Double_t aa, bbb, e, dedx;
// i=0;
Double_t E0=0.,Etem=0.,Esmall=0.,E2=0.,E=0.,dE=0.,x=0.,dx=0.25;
// Double_t E0=0.,Etem=0.,Esmall=0.,E2=0.,E=0.,dE=0.,x=0.,dx=0.25/20.;
Double_t sep[_N][NUM];

In [6]:
TString  lise_cal;
// lise_cal = TString::Format("/home/long/scripts/exp_scripts/lise/p_Si.txt");
lise_cal = TString::Format("/home/long/scripts/exp_scripts/lise/d_Si.txt");
int a_mass = 2; //deuteron

TGraph *g_1;

std::ifstream in_1(lise_cal.Data());

g_1 = new TGraph;

if(in_1.is_open())
{
    Int_t i = 0; 
    while(!in_1.eof())
    {
        in_1>>aa>>bbb>>e>>dedx>>aa>>bbb>>aa>>bbb>>aa>>bbb>>aa>>bbb>>aa>>bbb;
        
        g_1->SetPoint(i++, e*a_mass,dedx);    
    }
}
in_1.close();

c1->cd();
g_1->Draw();
c1->Draw();

In [7]:

Double_t d[_N-1];
Int_t k; //kind  0:Si 1:Al 2:Mylar
            
d[0]=0.;
d[1]=0.;
d[2]=0.;
d[3]=silicon_thickness;
d[4]=100000.;

for(int i=1;i<(_N-1);i++)
    {
        d[i]=d[i]+d[i-1];
        // std::cout << "d[" << i << "]=" << d[i] << "\n";
    }
for(int i=0;i<NUM;i++)   // Start simulation based on NUM events
{
    if(i%30==0)std::cout<<i<<":"<<NUM<<'\n';
    x=0;

    // First half of events NUM, energy from 0 to 40 MeV
    // if(i<=NUM)E0=E=i*200./(Double_t)(NUM);
    
    if(i<=NUM/2)E0=E=i*40./(Double_t)(NUM);
    else{E0=E=(40./2.) + (i-(Double_t)(NUM)/2)*220./(Double_t)(NUM);}

    for(Int_t j=0;j<_N;j++)sep[j][i]=0;
    sep[_N-1][i]=E0;
    Etem=E0;
    k=0;
    //cout<<" Y1"<<'\n';
    x=0.;
    while(1)
    {
        if((fabs(x-d[0])<=(dx/40.)||(x>=0&&x<d[0]))&&E<=0){sep[0][i]=Etem;Etem=0;break;}
        else if(fabs(x-d[0])<=(dx/40.)&&E>0){if(k!=0){sep[0][i]=E0-E;Etem=E;}k=0;}

        else if((fabs(x-d[1])<=(dx/2.)||(x>d[0]&&x<d[1]))&&E<=0){sep[1][i]=Etem;Etem=0;break;}
        else if(fabs(x-d[1])<=(dx/2.)&&E>0){if(k!=1){sep[1][i]=Etem-E;Etem=E;}k=1;}

        else if((fabs(x-d[2])<=(dx/2.)||(x>d[1]&&x<d[2]))&&E<=0){sep[2][i]=Etem;Etem=0;break;}
        else if(fabs(x-d[2])<=(dx/2.)&&E>0){if(k!=3){sep[2][i]=Etem-E;Etem=E;}k=3;}
        
        else if((fabs(x-d[3])<=(dx/40.)||(x>d[2]&&x<d[3]))&&E<=0){sep[3][i]=Etem;Etem=0;break;}
        else if(fabs(x-d[3])<=(dx/40.)&&E>0){if(k!=2){sep[3][i]=Etem-E;Etem=E;}k=2;}

        else if(((fabs(x-d[4])>(dx/40.)&&x>d[4])&&E<=0)){sep[4][i]=Etem;Etem=0;break;}
        else if(((fabs(x-d[4])>(dx/40.)&&x>d[4])&&E>0)){sep[4][i]=Etem-E;Etem=0;break;}
        else if(fabs(x-d[4])<=(dx/40.)&&E>0){if(k!=0){sep[4][i]=Etem-E;Etem=E;}k=0;}
        
        dE=g_1->Eval(E);
        E=E-dE*dx;
        x=x+dx;
    }
} 


0:100
30:100
60:100
90:100


In [8]:
TGraph *cal_pid;
cal_pid = new TGraph();

Int_t keypid=1;
Int_t ip=0,i03=0,i23=0,i43=0;

for(Int_t i0=0;i0<NUM;)
{
    if(keypid==1&&sep[4][i0]>0)cal_pid->SetPoint(ip++, sep[3][i0],sep[4][i0]);       
    i0++;
}
ip=0;
i03=0;
i23=0;
i43=0;
keypid=0;

c1->Clear();
cal_pid->SetMarkerStyle(3);
cal_pid->Draw("ap");
c1->Draw();

In [9]:
c1->Clear();
// gROOT->Macro("/home/long/data/25e04/10Be/hit/cut/cut_li6_L.C");
// gROOT->Macro("/home/long/data/25e04/10Be/hit/cut/cut_p_L.C");
gROOT->Macro("/home/long/data/25e04/10Be/hit/cut/cut_d_L.C");
TCutG *calib_cut [16];
for (int i = 0; i < 16; i++)
{
    // TString cut_name = TString::Format("calib_p_%i", i);
    TString cut_name = TString::Format("calib_d_%i", i);
    if(gROOT->GetListOfSpecials()->FindObject(cut_name))
    {
        std::cout << "cut_name = " << cut_name << "\n";
        calib_cut[i] = (TCutG *)gROOT->GetListOfSpecials()->FindObject(cut_name.Data());
    }
}
c1->Clear();

cut_name = calib_d_0


In [10]:
c1->Clear();

// double a = 20.;
// double b = -0.0107;

// double min_fit = 180.;
// double max_fit = 220.;
// double avg = (min_fit+max_fit)/2.;
// double avg_diff = avg-min_fit;

// min_fit = 183.;
// max_fit = 193.;
// avg = (min_fit+max_fit)/2.;
// avg_diff = avg-min_fit;

In [11]:
int strip_x_min = 0;
int strip_x_max = 3;
int strip_y_min = 12;
int strip_y_max = 15;
int cut_num = 0;

In [13]:
// TString name = TString::Format("(TMath::Sqrt(Rxc[0]*Rea[0]+%f*Rxc[0]*Rxc[0])+%f*Rea[0]):Rea[0]>>(1000,0,8000,1000,0.,400.)",a,b);
// TString name = TString::Format("(TMath::Sqrt(Rxc[0]*Rea[0]+%f*Rxc[0]*Rxc[0])+%f*Rea[0])>>(200,%f,%f)",a,b,min_fit,max_fit);
TString name = "Rxc[0]:Rea[0]:Rxc[0]>>(1000,0,8000,1000,0.,70.)";
// TString name = "Rxc[0]:Rea[0]>>h1";

TString draw_condition = TString::Format("Rxa_n[0]>=%i && Rxa_n[0]<=%i && Rya_n[0]>=%i && Rya_n[0]<=%i",strip_x_min, strip_x_max, strip_y_min, strip_y_max);
draw_condition = draw_condition + TString::Format("&& Rea_n[0]==%i", cut_num);

draw_condition = draw_condition + "&& af3>1500. && af3<3000. && ToF>170. && ToF<220.";
draw_condition = draw_condition + "&& xbt*xbt+ybt*ybt<100.";

// TString cut_name = TString::Format("calib_p_%i", cut_num);
TString cut_name = TString::Format("calib_d_%i", cut_num);
draw_condition = draw_condition + "&&" + cut_name.Data();

Int_t num1=fch->Draw(name.Data(),draw_condition.Data(),"");
c1->Draw();

if(num1==0) return;

In [14]:
TGraph2D *e1=new TGraph2D(num1,fch->GetV3(),fch->GetV2(),fch->GetV1());
Int_t npoints1=e1->GetN();

In [15]:
Double_t kk,bb;

Double_t *gex=e1->GetY(); // Energy deposit
Double_t *gey=e1->GetZ(); // delta E

In [32]:
Int_t J = 0;

// First make linear fit
TGraph *linear_result;
linear_result = new TGraph();

for(Int_t i=0;i<npoints1;i++){
   // if(gey[i]>0.1&&gex[i]>=200&&gex[i]<9000)
   // if(gey[i]>0.1&&gex[i]>=200&&gex[i]<4000)
   if(gey[i]>0.1&&gex[i]>=200)
   {
       double new_point = cal_pid->Eval(gey[i]);
       linear_result->SetPoint(J,new_point,gex[i]);
       J++;
   }
}

linear_result->Draw("ap");
c1->Draw();
TF1 *f1 = new TF1("f1","[1]*x+[0]",1.,100);
f1->SetParameter(0,90);
// f1->SetParameter(1,0);
linear_result->Fit("f1","rob=0.9");
kk=f1->GetParameter(1);
bb=f1->GetParameter(0);
cout<<"N="<<npoints1<<'\n';
cout<<"kp="<<kk<<" bp="<<bb<<'\n'; 
cout<<"chi2/NDF="<<f1->GetChisquare()/(Double_t)f1->GetNDF();

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =   1.1784e+08
NDf                       =         3797
Edm                       =  5.09583e-15
NCalls                    =           34
p0                        =       478.25   +/-   5.34619     
p1                        =      22.8264   +/-   0.116183    
N=3813
kp=22.8264 bp=478.25
chi2/NDF=31035.1

In [33]:
TGraph *func_result;
func_result = new TGraph();

Double_t a0,a1,a2;
c1->Clear();
J = 0;
for(Int_t i=0;i<npoints1;i++)
{
   // if(abs(gex[i]-(cal_pid->Eval(gey[i])*kk+bb))<5000) // difference between points and linear function
   {
       // if(gey[i]>0.1&&gex[i]>=200&&(gex[i]<=3000||(gex[i]>3000&&gex[i]<=6000&&i%10<=2)||(gex[i]>6000&&gex[i]<35000&&i%10<=1)))
       if(gey[i]>0.1 && gex[i]>=200 && gex[i]<=3500)
       // if(gey[i]>0.1 && gex[i]>=200)
       {
           func_result->SetPoint(J,cal_pid->Eval(gey[i]),gex[i]);
           J++;              
       }
   }
}
// TF1 *f3 = new TF1("f3","[0]*x-[0]*[1]*TMath::Log(1+x/[1])+[2]",1.,130);
TString fit_string = TString::Format("[0]*x^( ([1]+%i)/([2]+%i) )", a_mass, a_mass);
TF1 *f3 = new TF1("f3", fit_string,1.,130);
// f3->SetParameter(0,kk);
// f3->SetParLimits(1,0.,20.);
// f3->SetParameter(2,bb);

func_result->Draw("ap");
func_result->Fit("f3","", "", 10., 100.);
f3->Draw("same");

a0 = f3->GetParameter(0);
a1 = f3->GetParameter(1);
a2 = f3->GetParameter(2);
cout<<"a0="<<a0<<" a1="<<a1<<" a2="<<a2 << "\n";
cout<<"chi2/NDF="<<f3->GetChisquare()/(Double_t)f3->GetNDF();
c1->Draw();

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  4.77545e+07
NDf                       =         3405
Edm                       =  1.51884e-06
NCalls                    =          166
p0                        =      118.564   +/-   1.45824     
p1                        =      0.65683   +/-   0.263232    
p2                        =      1.89501   +/-   0.382096    
a0=118.564 a1=0.65683 a2=1.89501
chi2/NDF=14024.8

In [31]:
TGraph *func_result_modified;
func_result_modified = new TGraph();

c1->Clear();
J = 0;
for(Int_t i=0;i<npoints1;i++)
{
    double E = cal_pid->Eval(gey[i]);
    // if(abs( gex[i]-(a0*E-a0*a1*TMath::Log(1+E/a1)+a2) )<200) // difference between points and function
    if(abs( gex[i]-(a0*pow( E, (a1+a_mass)/(a2+a_mass) )) )<150) // difference between points and function
   {
       // if(gey[i]>0.1&&gex[i]>=200&&(gex[i]<=3000||(gex[i]>3000&&gex[i]<=6000&&i%10<=2)||(gex[i]>6000&&gex[i]<35000&&i%10<=1)))
       if(gey[i]>0.1 && gex[i]>=200 && gex[i]<=3500)
       // if(gey[i]>0.1 && gex[i]>=200)
       {
               func_result_modified->SetPoint(J,cal_pid->Eval(gey[i]),gex[i]);
               J++;              
       }
   }
}
func_result_modified->Draw("ap");
func_result_modified->Fit("f3","", "", 1., 100.);


// func_result->Draw("ap");

// f3->SetParameter(1,11.);
// f3->Draw("same");

a0 = f3->GetParameter(0);
a1 = f3->GetParameter(1);
a2 = f3->GetParameter(2);
cout<<"a0="<<a0<<" a1="<<a1<<" a2="<<a2 << "\n";
cout<<"chi2/NDF="<<f3->GetChisquare()/(Double_t)f3->GetNDF();
c1->Draw();

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  1.12818e+07
NDf                       =         3141
Edm                       =  4.57462e-08
NCalls                    =           91
p0                        =       108.78   +/-   0.708945    
p1                        =     0.700009   +/-   0.204098    
p2                        =      1.83174   +/-   0.288286    
a0=108.78 a1=0.700009 a2=1.83174
chi2/NDF=3591.78